# Felix Mumo - Assignment 11
## BERT Embeddings and Cosine Similarity

In this assignment I use a pre-trained BERT model (bert-base-uncased) from Hugging Face to turn sentences into embeddings. I compare 10 sentence pairs using cosine similarity, predict whether they are similar with a threshold of 0.7, and calculate the accuracy against my own labels. I also look at token-level embeddings to show how BERT gives the same word a different meaning depending on its context.

### 1. Import libraries and load BERT

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics import accuracy_score

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()

print("Model loaded")

### 2. Token-level embeddings
BERT splits a sentence into tokens and adds two special tokens, [CLS] at the start and [SEP] at the end. Each token gets its own vector of 768 numbers.

In [ ]:
sentence = "How do I learn Python?"
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
token_embeddings = outputs.last_hidden_state[0]

print("Tokens:", tokens)
print("Token embeddings shape:", token_embeddings.shape)
print("First 5 values of the embedding for 'python':", token_embeddings[tokens.index("python")][:5].numpy())

### 3. Functions for sentence embeddings and cosine similarity
To get one vector for the whole sentence, I take the mean of all the token embeddings (mean pooling).

Cosine similarity measures the angle between two vectors:

cosine similarity = (A . B) / (||A|| x ||B||)

A value close to 1 means the vectors point in the same direction (similar meaning).

In [ ]:
def get_sentence_embedding(sentence):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0].mean(dim=0).numpy()


def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

### 4. Polysemy: the same word in different contexts
The word "bank" can mean a place for money or the side of a river. I take the embedding of the token "bank" from three sentences and compare them.

In [ ]:
def get_word_embedding(sentence, word):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    return outputs.last_hidden_state[0][tokens.index(word)].numpy()


bank_sentences = [
    "I deposited money at the bank.",
    "The bank approved my loan application.",
    "We had a picnic on the river bank.",
]
bank_vectors = [get_word_embedding(s, "bank") for s in bank_sentences]

print("money bank vs loan bank: ", round(cosine_sim(bank_vectors[0], bank_vectors[1]), 4))
print("money bank vs river bank:", round(cosine_sim(bank_vectors[0], bank_vectors[2]), 4))
print("loan bank vs river bank: ", round(cosine_sim(bank_vectors[1], bank_vectors[2]), 4))

In [ ]:
labels = ["bank (money)", "bank (loan)", "bank (river)"]
matrix = [[cosine_sim(a, b) for b in bank_vectors] for a in bank_vectors]

plt.figure(figsize=(6, 4.5))
sns.heatmap(matrix, annot=True, fmt=".2f", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Cosine similarity of the word 'bank' in different sentences")
plt.show()

The two financial meanings of "bank" are more similar to each other than to the river meaning. A static word vector (like Word2Vec or GloVe) would give "bank" the exact same vector in all three sentences, so all the similarities would be 1.0. BERT changes the vector based on the surrounding words.

I did the same with the word "apple" (the fruit and the company):

In [ ]:
apple_sentences = [
    "I ate a red apple for breakfast.",
    "She cut the apple into small pieces.",
    "Apple released a new iPhone this year.",
]
apple_vectors = [get_word_embedding(s, "apple") for s in apple_sentences]

print("fruit vs fruit:  ", round(cosine_sim(apple_vectors[0], apple_vectors[1]), 4))
print("fruit vs company:", round(cosine_sim(apple_vectors[0], apple_vectors[2]), 4))

### 5. Sentence pairs
The first 5 pairs were given. I added 5 more of my own: some with similar meaning, some unrelated, and one tricky pair that shares the word "bank" but has a different meaning.

My ground truth labels: 1 = similar, 0 = not similar.

In [ ]:
sentence_pairs = [
    ("How do I learn Python?", "What is the best way to study Python?"),
    ("What is AI?", "How to cook pasta?"),
    ("How do I bake a chocolate cake?", "Give me a chocolate cake recipe."),
    ("How can I improve my coding skills?", "Tips for becoming better at programming."),
    ("Where can I buy cheap laptops?", "Best sites to find affordable computers."),
    # my own pairs
    ("How do I open a bank account?", "What are the steps to create a new bank account?"),
    ("The weather is sunny today.", "I need to fix my car engine."),
    ("What is the capital city of Kenya?", "Which city is the capital of Kenya?"),
    ("I sat on the river bank.", "I withdrew money from the bank."),
    ("How do I reset my password?", "What time does the football match start?"),
]

true_labels = [1, 0, 1, 1, 1, 1, 0, 1, 0, 0]

### 6. Compute similarity and predictions (threshold 0.7)

In [ ]:
threshold = 0.7
results = []

for (s1, s2), actual in zip(sentence_pairs, true_labels):
    similarity = cosine_sim(get_sentence_embedding(s1), get_sentence_embedding(s2))
    predicted = 1 if similarity > threshold else 0
    results.append({
        "Sentence 1": s1,
        "Sentence 2": s2,
        "Similarity": round(float(similarity), 4),
        "Predicted": predicted,
        "Actual": actual,
        "Correct": predicted == actual,
    })

results_df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", None)
results_df

### 7. Accuracy

In [ ]:
accuracy = accuracy_score(results_df["Actual"], results_df["Predicted"])
print(f"Accuracy: {accuracy * 100:.0f}% ({results_df['Correct'].sum()} out of {len(results_df)} correct)")

print("\nPairs predicted wrongly:")
print(results_df.loc[~results_df["Correct"], ["Sentence 1", "Sentence 2", "Similarity", "Actual"]].to_string(index=False))

In [ ]:
colors = ["tab:green" if a == 1 else "tab:red" for a in results_df["Actual"]]

plt.figure(figsize=(9, 4.5))
plt.bar(range(1, 11), results_df["Similarity"], color=colors)
plt.axhline(threshold, color="black", linestyle="--", label="Threshold 0.7")
plt.xticks(range(1, 11), [f"Pair {i}" for i in range(1, 11)], rotation=45)
plt.ylabel("Cosine similarity")
plt.title("BERT similarity per pair (green = actually similar, red = not similar)")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

**Observation:** with plain BERT, even unrelated sentences get fairly high similarity scores. This is because BERT was not trained to compare sentences, and its embeddings all point in a similar general direction. So a fixed threshold of 0.7 can wrongly mark some unrelated pairs as similar. The similar pairs still score higher than the unrelated ones, so a higher threshold would separate them better.

In [ ]:
for t in [0.7, 0.75, 0.8, 0.85, 0.9]:
    preds = (results_df["Similarity"] > t).astype(int)
    print(f"Threshold {t}: accuracy = {accuracy_score(results_df['Actual'], preds) * 100:.0f}%")

### 8. Bonus: comparison with Sentence-BERT
Sentence-BERT (all-MiniLM-L6-v2) is a version of BERT that was fine-tuned specifically to compare sentences. I ran the same 10 pairs through it with the same threshold of 0.7.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer("all-MiniLM-L6-v2")

sbert_scores = []
for s1, s2 in sentence_pairs:
    e1, e2 = sbert.encode([s1, s2])
    sbert_scores.append(round(float(cosine_sim(e1, e2)), 4))

results_df["SBERT Similarity"] = sbert_scores
results_df["SBERT Predicted"] = (results_df["SBERT Similarity"] > threshold).astype(int)

print(f"BERT accuracy:  {accuracy_score(results_df['Actual'], results_df['Predicted']) * 100:.0f}%")
print(f"SBERT accuracy: {accuracy_score(results_df['Actual'], results_df['SBERT Predicted']) * 100:.0f}%")
results_df[["Sentence 1", "Sentence 2", "Actual", "Similarity", "Predicted", "SBERT Similarity", "SBERT Predicted"]]

Sentence-BERT gives a much wider range of scores: unrelated pairs get low scores and similar pairs get high scores, so the 0.7 threshold works better with it.

### 9. Questions

**How does BERT differ from traditional NLP approaches?**

Traditional approaches like Bag of Words and TF-IDF only count words, so they ignore word order and meaning. Static word embeddings like Word2Vec and GloVe give each word one fixed vector, so a word like "bank" has the same vector whether it is about money or a river. BERT is a transformer model that reads the whole sentence in both directions at once using self-attention. Because of this, each word gets a contextual embedding that depends on the words around it. BERT is also pre-trained on a very large amount of text, so it can be reused for many tasks such as classification, question answering and similarity.

**What is cosine similarity, and why is it useful in comparing embeddings?**

Cosine similarity measures the cosine of the angle between two vectors. It is calculated as the dot product of the vectors divided by the product of their lengths, and it ranges from -1 to 1, where 1 means they point in exactly the same direction. It is useful for embeddings because it compares the direction of the vectors (the meaning) and not their size, so sentences of different lengths can still be compared fairly. It also works well with high-dimensional vectors like BERT's 768 values and is quick to calculate.

### 10. Conclusion
- BERT produces a 768-value embedding for every token, and the embedding changes with context. The word "bank" had a lower similarity between its river and money meanings than between its two money meanings.
- Using mean pooling and cosine similarity, I compared 10 sentence pairs and calculated the accuracy at a threshold of 0.7 (see section 7).
- Plain BERT gives high similarity scores even to unrelated sentences, so 0.7 is too low a threshold for it. A higher threshold, or a model trained for sentence similarity like Sentence-BERT, gives better results.